# `tensorium` tutorial, part 3: tensor fields with internal values


In [1]:
from sympy import simplify, symbols
from tensorium import *

S2 = Manifold("S^2", 2)
U_N = OpenSet("U_N", S2)  # S^2 without the north pole
U_S = OpenSet("U_S", S2)  # S^2 without the south pole
x, y = symbols("x y", real=True)
u, v = symbols("u v", real=True)
X_N = Chart("X_N", U_N, (x, y))
X_S = Chart("X_S", U_S, (u, v), relations={X_N: (u/(u**2 + v**2), v/(u**2 + v**2))})
atlas = Atlas(S2, [X_N, X_S])
S2.set_atlas(atlas)
xN, yN = X_N.symbols
uS, vS = X_S.symbols
f_N_local = LocalTensorField(X_N, (0, 0), 1/(1 + xN**2 + yN**2))
f_S_local = LocalTensorField(X_S, (0, 0), (uS**2 + vS**2)/(1 + uS**2 + vS**2))
f = TensorField(S2, (0, 0), {X_N: f_N_local, X_S: f_S_local}, ())
V_N_local = LocalTensorField(X_N, (1, 0), [-yN, xN])
V_S_local = LocalTensorField(X_S, (1, 0), [-vS, uS])
V = Vector(S2, {X_N: V_N_local, X_S: V_S_local})
omega_N_local = LocalTensorField(X_N, (0, 1), [-2*xN/(1 + xN**2 + yN**2)**2, -2*yN/(1 + xN**2 + yN**2)**2])
omega_S_local = LocalTensorField(X_S, (0, 1), [2*uS/(1 + uS**2 + vS**2)**2, 2*vS/(1 + uS**2 + vS**2)**2])
omega = OneForm(S2, {X_N: omega_N_local, X_S: omega_S_local})
W_N_local = LocalTensorField(X_N, (1, 0), [xN, yN])
W_S_local = LocalTensorField(X_S, (1, 0), [-uS, -vS])
W = Vector(S2, {X_N: W_N_local, X_S: W_S_local})
T = V * omega


## 3. Valued tensor fields


So far every index was geometric: it referred to directions on the manifold. Many physical fields also carry **internal indices**. 
$$
A\in \mathcal{T}^r_s(M)\otimes E_n\otimes E_m^*
$$

is a tensor field on $M$ whose components are valued in a finite-dimensional internal tensor product. This is a trivialized/local description. The library does not model general non-trivial vector bundles or changes of internal frame.

The library separates the two structures:

- geometric indices are handled by `TensorField`, `Vector`, `OneForm`, metrics and affine connections;
- internal indices are handled by `TensorMultiplet` and `ValuedTensorField`.

Coordinate changes on $M$ act only on the geometric tensor components. Internal indices are transformed only by explicitly provided internal operations, such as internal contractions or gauge connections.

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">

Valued fields also carry a lightweight <code>FieldSignature</code>. This signature is used by tensor operators to reject incompatible inputs before doing component-level computations.

</div>

### 3.1. Internal vectors: tensor multiplets


A `TensorMultiplet` is the simplest field with internal values: a tensor field with one internal index in a fixed internal vector space. Below we build a two-component scalar multiplet

$$
\psi^{[A]}=(f,2f),\qquad A=0,1.
$$

Each component $\psi^{[A]}$ is still an ordinary scalar field on $S^2$. The bracketed index labels the chosen basis of the internal space $E_2$; it is not a coordinate index on the sphere.



In [2]:
psi = TensorMultiplet([f, 2*f], internal_variance=1)

Display(psi, name=r"\psi")

<IPython.core.display.Math object>

A dual internal multiplet has a covariant internal index. We can contract it with <code>psi</code> using <code>internal_action</code>, which automatically detects the unique pair of opposite-variance internal indices.</p>


In [3]:
chi = TensorMultiplet([3*f, -f], internal_variance=-1)
chi_on_psi = chi.internal_action(psi)

Display(chi, name=r"\chi")
Display(chi_on_psi, name=r"\chi(\psi)")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 3.2. Matrix-valued fields and internal actions


The general class `ValuedTensorField` can carry several internal indices. A matrix-valued scalar field has one contravariant and one covariant internal index. The two slots do not need to have the same dimension unless we want an endomorphism.

Here we build a field-valued linear map

$$
Q^{[A]}{}_{[I]}:F\longrightarrow E,
\qquad \dim E=2,\quad \dim F=3,
$$

represented as

$$
Q\in C^\infty(S^2)\otimes E_2\otimes E_3^*.
$$

Acting on an internal vector $\eta^{[I]}\in E_3$ contracts only the matching three-dimensional slot:

$$
(Q\eta)^{[A]}=\sum_{I=0}^{2}Q^{[A]}{}_{[I]}\eta^{[I]}.
$$

This is useful when different physical indices belong to different finite-dimensional internal spaces.


In [4]:
one = TensorField(S2, (0, 0), {
    X_N: LocalTensorField(X_N, (0, 0), 1),
    X_S: LocalTensorField(X_S, (0, 0), 1),
}, ())
zero = 0*one

Q = ValuedTensorField([[one, zero, 2*one], [-one, one, zero]],
                        internal_shape=(2, 3), internal_variance=(1, -1))
eta = ValuedTensorField([one, 2*one, -one],
                          internal_shape=(3,), internal_variance=(1,))
Qeta = Q.internal_action(eta)

Display(Q, name="Q")
Display(eta, name=r"\eta")
Display(Qeta, name=r"Q\eta")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<!-- small-note-html -->
<div style="font-size:0.82em; line-height:1.35; padding:0.45em 0.70em; margin:0.45em 0 0.70em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
  <p style="margin:0.18em 0;">
  An <code>ValuedTensorField</code> receives its internal components as a nested Python list or tuple. The nesting must match <code>internal_shape</code>. For example, <code>internal_shape=(2,3)</code> expects a list with two rows and three entries per row, such as <code>[[T00,T01,T02],[T10,T11,T12]]</code>.
  </p>
</div>

<!-- small-note-html -->
<div style="font-size:0.82em; line-height:1.35; padding:0.45em 0.70em; margin:0.45em 0 0.70em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
  <p style="margin:0.18em 0;">
  The method <code>internal_action</code> forms the tensor product and then contracts compatible internal indices. If there is a unique compatible pair, the library infers it automatically. Otherwise, the contraction pairs must be supplied explicitly.
  </p>
</div>

### 3.3. External products of internal fields


The tensor product preserves both structures at once. If we multiply an internal vector by an internal covector, the result carries two internal indices:

$$
(\psi\otimes\chi)^{[A]}{}_{[B]}\in C^\infty(S^2)\otimes E_2\otimes E_2^*.
$$

At the same time, the geometric tensor product is applied to the tensor-field components.



In [5]:
projector = psi.tensor_product(chi)

Display(projector, name=r"(\psi\otimes\chi)")

<IPython.core.display.Math object>

Internal indices and geometric indices are independent. For instance, we can multiply the scalar multiplet by a one-form and obtain a one-form-valued internal vector,
$$
\psi\otimes\omega\in \mathcal{T}^0_1(S^2)\otimes E_2.
$$
The internal index records the $E_2$ component, while the geometric covariant index records the one-form component.


In [6]:
psi_omega = psi.tensor_product(omega)

Display(psi_omega, name=r"(\psi\otimes\omega)")

<IPython.core.display.Math object>